# Algorithmes d'optimisation sans contrainte — implémentation *from scratch*

**Librairies autorisées : `numpy` et `sympy` uniquement.**

Pour chaque algorithme : la théorie, l'implémentation, puis un exemple déroulé.

| # | Algorithme |
|---|---|
| 1 | Descente de gradient à pas fixe |
| 2 | Descente de gradient à pas variable (Armijo) |
| 3 | Gradient conjugué (cas quadratique, puis non linéaire) |
| 4 | Quasi-Newton (BFGS) |

In [1]:
from dataclasses import dataclass, field

import numpy as np
import sympy as sp

np.set_printoptions(precision=6, suppress=True)
x, y = sp.symbols("x y")


def titre(texte):
    print("\n" + "=" * 80)
    print(texte)
    print("=" * 80)

## 0. Outils communs

- `Resultat` : conteneur pour la solution, le nombre d'itérations, l'historique…
- `fonction_depuis_sympy` : à partir d'une expression sympy, construit `f(x)` et `∇f(x)` (le gradient est calculé **symboliquement** par sympy).
- `backtracking_armijo` : recherche linéaire. On cherche `α` tel que

  `f(x + α d) ≤ f(x) + c · α · ⟨∇f(x), d⟩`   (condition d'Armijo)

  en partant de `α₀` et en multipliant par `ρ = 0.5` tant que la condition échoue.
- `_affiche_entete` / `_affiche_ligne` : tableau des itérations.

In [2]:
@dataclass
class Resultat:
    x: np.ndarray                 # dernier itéré
    f: float                      # f(x)
    norme_grad: float             # ||grad f(x)||
    n_iter: int                   # nombre d'itérations effectuées
    converge: bool                # critère d'arrêt atteint ?
    historique: list = field(default_factory=list)  # liste des x_k
    H_inverse: np.ndarray = None  # approximation de l'inverse de la Hessienne (BFGS)


def fonction_depuis_sympy(expr, variables):
    """
    Construit f(x) et grad f(x) (fonctions numpy) à partir d'une expression
    sympy. Le gradient est calculé symboliquement par sympy.

    >>> x, y = sp.symbols("x y")
    >>> f, grad = fonction_depuis_sympy(x**2 + y**2, [x, y])
    >>> f(np.array([1.0, 2.0]))
    5.0
    """
    f_num = sp.lambdify(variables, expr, "numpy")
    grad_sym = [sp.diff(expr, v) for v in variables]
    grad_num = sp.lambdify(variables, grad_sym, "numpy")

    def f(x):
        return float(f_num(*x))

    def grad(x):
        return np.array(grad_num(*x), dtype=float)

    return f, grad


def backtracking_armijo(f, g_x, x, d, alpha0=1.0, c=1e-4, rho=0.5, max_iter=60):
    """
    Cherche alpha tel que  f(x + alpha d) <= f(x) + c * alpha * <grad f(x), d>.
    On part de alpha0 et on le multiplie par rho tant que la condition échoue.
    """
    fx = f(x)
    pente = float(g_x @ d)       # doit être < 0 pour une direction de descente
    alpha = alpha0
    for _ in range(max_iter):
        if f(x + alpha * d) <= fx + c * alpha * pente:
            return alpha
        alpha *= rho
    return alpha


def _affiche_entete(titre):
    print(f"\n{titre}")
    print(f"{'k':>4} | {'x_k':<28} | {'f(x_k)':>14} | {'||grad||':>11} | {'alpha_k':>10}")
    print("-" * 80)


def _affiche_ligne(k, x, fx, ng, alpha):
    xs = np.array2string(x, precision=5, suppress_small=True)
    a = "-" if alpha is None else f"{alpha:.5f}"
    print(f"{k:>4} | {xs:<28} | {fx:>14.8f} | {ng:>11.3e} | {a:>10}")

## Fonction de test n°1 : une quadratique

$$f(x,y) = 2x^2 + 1.5y^2 + xy - x - 2y \;=\; \tfrac12 X^T A X - b^T X,\qquad A=\begin{pmatrix}4&1\\1&3\end{pmatrix},\; b=\begin{pmatrix}1\\2\end{pmatrix}$$

Minimum exact : $X^* = A^{-1}b = (1/11,\, 7/11)$.

In [3]:
A = np.array([[4.0, 1.0], [1.0, 3.0]])
b = np.array([1.0, 2.0])
x_exact = np.linalg.solve(A, b)

expr_quad = 2 * x**2 + sp.Rational(3, 2) * y**2 + x * y - x - 2 * y
f_quad, grad_quad = fonction_depuis_sympy(expr_quad, [x, y])

print("Gradient symbolique :", [sp.diff(expr_quad, v) for v in (x, y)])
print("Solution exacte     :", x_exact)

Gradient symbolique : [4*x + y - 1, x + 3*y - 2]
Solution exacte     : [0.090909 0.636364]


## 1. Descente de gradient à pas fixe

**Principe** : on avance à chaque itération dans la direction opposée au gradient, avec un pas constant :

$$x_{k+1} = x_k - \alpha\,\nabla f(x_k)$$

**Arrêt** : $\|\nabla f(x_k)\| < \text{tol}$.

**Convergence** (cas quadratique) : garantie si $0 < \alpha < 2/\lambda_{\max}(A)$.

In [4]:
def descente_gradient_pas_fixe(f, grad, x0, alpha=0.1, tol=1e-6,
                               max_iter=10_000, verbose=False, n_affiche=10):
    """
    x_{k+1} = x_k - alpha * grad f(x_k)
    Arrêt quand ||grad f(x_k)|| < tol.
    Convergence garantie (cas quadratique) si 0 < alpha < 2 / lambda_max(H).
    """
    x = np.array(x0, dtype=float)
    hist = [x.copy()]
    if verbose:
        _affiche_entete(f"Descente de gradient - pas fixe (alpha = {alpha})")

    for k in range(max_iter):
        g = grad(x)
        ng = np.linalg.norm(g)
        if verbose and k < n_affiche:
            _affiche_ligne(k, x, f(x), ng, alpha)
        if ng < tol:
            if verbose:
                _affiche_ligne(k, x, f(x), ng, None)
            return Resultat(x, f(x), ng, k, True, hist)
        x = x - alpha * g
        hist.append(x.copy())

    g = grad(x)
    return Resultat(x, f(x), np.linalg.norm(g), max_iter, False, hist)

### Exemple 1 — pas fixe

$X_0=(0,0)$, $\alpha = 0.2$. On déroule l'itération 0 à la main, puis on lance l'algorithme.

In [5]:
titre("EXEMPLE 1 - Descente de gradient a PAS FIXE")
print("f(x,y) = 2x^2 + 1.5y^2 + xy - x - 2y,   X0 = (0, 0),   alpha = 0.2")
print("grad f = (4x + y - 1 ,  x + 3y - 2)")
print("Condition de convergence : alpha < 2/lambda_max(A) =",
      round(2 / max(np.linalg.eigvalsh(A)), 4))

print("\n--- Deroulement a la main de l'iteration 0 ---")
X0 = np.array([0.0, 0.0])
g0 = grad_quad(X0)
X1 = X0 - 0.2 * g0
print(f"grad f(X0) = {g0}")
print(f"X1 = X0 - 0.2 * grad f(X0) = {X0} - 0.2 * {g0} = {X1}")
print(f"f(X0) = {f_quad(X0):.4f}  ->  f(X1) = {f_quad(X1):.4f}")

res = descente_gradient_pas_fixe(f_quad, grad_quad, X0, alpha=0.2,
                                 tol=1e-6, verbose=True, n_affiche=8)
print(f"\nResultat : X = {res.x}  en {res.n_iter} iterations")
print(f"Solution exacte : {x_exact}")


EXEMPLE 1 - Descente de gradient a PAS FIXE
f(x,y) = 2x^2 + 1.5y^2 + xy - x - 2y,   X0 = (0, 0),   alpha = 0.2
grad f = (4x + y - 1 ,  x + 3y - 2)
Condition de convergence : alpha < 2/lambda_max(A) = 0.4331

--- Deroulement a la main de l'iteration 0 ---
grad f(X0) = [-1. -2.]
X1 = X0 - 0.2 * grad f(X0) = [0. 0.] - 0.2 * [-1. -2.] = [0.2 0.4]
f(X0) = 0.0000  ->  f(X1) = -0.6000

Descente de gradient - pas fixe (alpha = 0.2)
   k | x_k                          |         f(x_k) |    ||grad|| |    alpha_k
--------------------------------------------------------------------------------
   0 | [0. 0.]                      |     0.00000000 |   2.236e+00 |    0.20000
   1 | [0.2 0.4]                    |    -0.60000000 |   6.325e-01 |    0.20000
   2 | [0.16 0.52]                  |    -0.66000000 |   3.225e-01 |    0.20000
   3 | [0.128 0.576]                |    -0.67584000 |   1.688e-01 |    0.20000
   4 | [0.1104 0.6048]              |    -0.68017920 |   8.836e-02 |    0.20000
   5 | [0.

## 2. Descente de gradient à pas variable (Armijo)

**Principe** : même direction $d_k=-\nabla f(x_k)$, mais le pas $\alpha_k$ est choisi à chaque itération par **backtracking d'Armijo** :

$$x_{k+1} = x_k + \alpha_k d_k,\qquad f(x_k+\alpha_k d_k) \le f(x_k) + c\,\alpha_k \langle\nabla f(x_k), d_k\rangle$$

Plus besoin de connaître $\lambda_{\max}$ à l'avance.

In [6]:
def descente_gradient_pas_variable(f, grad, x0, alpha0=1.0, tol=1e-6,
                                   max_iter=10_000, verbose=False, n_affiche=10):
    """
    x_{k+1} = x_k - alpha_k * grad f(x_k)
    où alpha_k est choisi à chaque itération par backtracking d'Armijo.
    """
    x = np.array(x0, dtype=float)
    hist = [x.copy()]
    if verbose:
        _affiche_entete("Descente de gradient - pas variable (Armijo)")

    for k in range(max_iter):
        g = grad(x)
        ng = np.linalg.norm(g)
        if ng < tol:
            if verbose:
                _affiche_ligne(k, x, f(x), ng, None)
            return Resultat(x, f(x), ng, k, True, hist)
        d = -g
        alpha = backtracking_armijo(f, g, x, d, alpha0)
        if verbose and k < n_affiche:
            _affiche_ligne(k, x, f(x), ng, alpha)
        x = x + alpha * d
        hist.append(x.copy())

    g = grad(x)
    return Resultat(x, f(x), np.linalg.norm(g), max_iter, False, hist)

### Exemple 2 — pas variable

Même fonction, $X_0=(0,0)$, $\alpha_0=1$, $c=10^{-4}$, $\rho=0.5$. On voit le backtracking à l'œuvre à l'itération 0.

In [7]:
titre("EXEMPLE 2 - Descente de gradient a PAS VARIABLE (Armijo)")
print("Meme fonction, X0 = (0, 0), alpha0 = 1, c = 1e-4, rho = 0.5")

print("\n--- Deroulement a la main de l'iteration 0 ---")
X0 = np.array([0.0, 0.0])
g0 = grad_quad(X0)
d = -g0
print(f"grad f(X0) = {g0},  direction d0 = {d},  pente <g,d> = {g0 @ d}")
alpha = 1.0
while f_quad(X0 + alpha * d) > f_quad(X0) + 1e-4 * alpha * (g0 @ d):
    print(f"  alpha = {alpha:<6} : f(X0+alpha d) = {f_quad(X0 + alpha * d):.4f} "
          f"> seuil Armijo -> on divise par 2")
    alpha *= 0.5
print(f"  alpha = {alpha:<6} : f(X0+alpha d) = {f_quad(X0 + alpha * d):.4f} "
      f"<= seuil Armijo -> accepte")

res = descente_gradient_pas_variable(f_quad, grad_quad, X0,
                                     tol=1e-6, verbose=True, n_affiche=8)
print(f"\nResultat : X = {res.x}  en {res.n_iter} iterations")
print(f"Solution exacte : {x_exact}")


EXEMPLE 2 - Descente de gradient a PAS VARIABLE (Armijo)
Meme fonction, X0 = (0, 0), alpha0 = 1, c = 1e-4, rho = 0.5

--- Deroulement a la main de l'iteration 0 ---
grad f(X0) = [-1. -2.],  direction d0 = [1. 2.],  pente <g,d> = -5.0
  alpha = 1.0    : f(X0+alpha d) = 5.0000 > seuil Armijo -> on divise par 2
  alpha = 0.5    : f(X0+alpha d) = 0.0000 > seuil Armijo -> on divise par 2
  alpha = 0.25   : f(X0+alpha d) = -0.6250 <= seuil Armijo -> accepte

Descente de gradient - pas variable (Armijo)
   k | x_k                          |         f(x_k) |    ||grad|| |    alpha_k
--------------------------------------------------------------------------------
   0 | [0. 0.]                      |     0.00000000 |   2.236e+00 |    0.25000
   1 | [0.25 0.5 ]                  |    -0.62500000 |   5.590e-01 |    0.50000
   2 | [0.    0.625]                |    -0.66406250 |   3.953e-01 |    0.25000
   3 | [0.09375 0.65625]            |    -0.68115234 |   6.988e-02 |    0.25000
   4 | [0.08594 

## 3. Gradient conjugué

### 3a. Cas quadratique $f(x)=\tfrac12 x^TAx-b^Tx$

On construit des directions **A-conjuguées** ($d_i^T A d_j = 0$). Avec $r_k = b - Ax_k = -\nabla f(x_k)$ et $d_0=r_0$ :

$$\alpha_k=\frac{r_k^Tr_k}{d_k^TAd_k},\quad x_{k+1}=x_k+\alpha_k d_k,\quad r_{k+1}=r_k-\alpha_k A d_k$$
$$\beta_k=\frac{r_{k+1}^Tr_{k+1}}{r_k^Tr_k},\quad d_{k+1}=r_{k+1}+\beta_k d_k$$

Le pas est **exact** et la méthode converge en **au plus $n$ itérations**.

In [8]:
def gradient_conjugue_quadratique(A, b, x0, tol=1e-10, max_iter=None,
                                  verbose=False):
    """
    Gradient conjugué linéaire (A symétrique définie positive).
    Pas optimal exact : alpha_k = <r_k, r_k> / <d_k, A d_k>
    Converge en au plus n itérations (n = dimension) en arithmétique exacte.
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    x = np.array(x0, dtype=float)
    n = len(b)
    max_iter = n if max_iter is None else max_iter

    f = lambda z: 0.5 * z @ A @ z - b @ z
    r = b - A @ x                 # r = -grad f(x)
    d = r.copy()
    hist = [x.copy()]
    if verbose:
        _affiche_entete("Gradient conjugue (quadratique, pas exact)")

    for k in range(max_iter + 1):
        nr = np.linalg.norm(r)
        if nr < tol:
            if verbose:
                _affiche_ligne(k, x, f(x), nr, None)
            return Resultat(x, f(x), nr, k, True, hist)
        if k == max_iter:
            break
        Ad = A @ d
        alpha = (r @ r) / (d @ Ad)
        if verbose:
            _affiche_ligne(k, x, f(x), nr, alpha)
        x = x + alpha * d
        r_new = r - alpha * Ad
        beta = (r_new @ r_new) / (r @ r)          # Fletcher-Reeves
        d = r_new + beta * d
        r = r_new
        hist.append(x.copy())

    return Resultat(x, f(x), np.linalg.norm(r), max_iter, False, hist)

### 3b. Cas non linéaire (Polak-Ribière+)

Pour une fonction quelconque, le pas exact n'existe plus : on utilise Armijo.

$$d_0=-g_0,\quad x_{k+1}=x_k+\alpha_k d_k,\quad \beta_k=\max\!\Big(0,\frac{g_{k+1}^T(g_{k+1}-g_k)}{g_k^Tg_k}\Big),\quad d_{k+1}=-g_{k+1}+\beta_k d_k$$

Redémarrage ($d=-g$) si $d$ n'est pas une direction de descente.

In [9]:
def gradient_conjugue_non_lineaire(f, grad, x0, tol=1e-6, max_iter=10_000,
                                   verbose=False, n_affiche=10):
    """
    d_0 = -g_0 ;  x_{k+1} = x_k + alpha_k d_k  (alpha_k par Armijo)
    beta_k = max(0, <g_{k+1}, g_{k+1} - g_k> / <g_k, g_k>)    (Polak-Ribière+)
    d_{k+1} = -g_{k+1} + beta_k d_k
    Redémarrage (d = -g) si d n'est pas une direction de descente.
    """
    x = np.array(x0, dtype=float)
    g = grad(x)
    d = -g
    hist = [x.copy()]
    if verbose:
        _affiche_entete("Gradient conjugue non lineaire (Polak-Ribiere+)")

    for k in range(max_iter):
        ng = np.linalg.norm(g)
        if ng < tol:
            if verbose:
                _affiche_ligne(k, x, f(x), ng, None)
            return Resultat(x, f(x), ng, k, True, hist)
        if g @ d >= 0:                       # sécurité : redémarrage
            d = -g
        alpha = backtracking_armijo(f, g, x, d)
        if verbose and k < n_affiche:
            _affiche_ligne(k, x, f(x), ng, alpha)
        x = x + alpha * d
        g_new = grad(x)
        beta = max(0.0, g_new @ (g_new - g) / (g @ g))
        d = -g_new + beta * d
        g = g_new
        hist.append(x.copy())

    return Resultat(x, f(x), np.linalg.norm(g), max_iter, False, hist)

### Exemple 3 — gradient conjugué

Sur la quadratique : convergence en $n=2$ itérations pour la version exacte, puis version non linéaire.

In [10]:
titre("EXEMPLE 3 - GRADIENT CONJUGUE")
print("--- 3a. Cas quadratique : convergence en n = 2 iterations ---")
X0 = np.array([0.0, 0.0])
r0 = b - A @ X0
alpha0 = (r0 @ r0) / (r0 @ A @ r0)
print(f"r0 = b - A X0 = {r0} = d0")
print(f"alpha0 = <r0,r0> / <d0, A d0> = {alpha0:.6f}")
print(f"X1 = X0 + alpha0 d0 = {X0 + alpha0 * r0}")

res = gradient_conjugue_quadratique(A, b, X0, verbose=True)
print(f"\nResultat : X = {res.x}  en {res.n_iter} iterations (n = 2)")
print(f"Solution exacte : {x_exact}")

print("\n--- 3b. Cas non lineaire (Polak-Ribiere+) sur la fonction quadratique ---")
res = gradient_conjugue_non_lineaire(f_quad, grad_quad, X0, verbose=True)
print(f"\nResultat : X = {res.x}  en {res.n_iter} iterations")


EXEMPLE 3 - GRADIENT CONJUGUE
--- 3a. Cas quadratique : convergence en n = 2 iterations ---
r0 = b - A X0 = [1. 2.] = d0
alpha0 = <r0,r0> / <d0, A d0> = 0.250000
X1 = X0 + alpha0 d0 = [0.25 0.5 ]

Gradient conjugue (quadratique, pas exact)
   k | x_k                          |         f(x_k) |    ||grad|| |    alpha_k
--------------------------------------------------------------------------------
   0 | [0. 0.]                      |     0.00000000 |   2.236e+00 |    0.25000
   1 | [0.25 0.5 ]                  |    -0.62500000 |   5.590e-01 |    0.36364
   2 | [0.09091 0.63636]            |    -0.68181818 |   0.000e+00 |          -

Resultat : X = [0.090909 0.636364]  en 2 iterations (n = 2)
Solution exacte : [0.090909 0.636364]

--- 3b. Cas non lineaire (Polak-Ribiere+) sur la fonction quadratique ---

Gradient conjugue non lineaire (Polak-Ribiere+)
   k | x_k                          |         f(x_k) |    ||grad|| |    alpha_k
-------------------------------------------------------

## 4. Quasi-Newton (BFGS)

La méthode de Newton utilise $d_k=-[\nabla^2 f(x_k)]^{-1}\nabla f(x_k)$ mais la Hessienne est coûteuse. **Quasi-Newton** maintient une approximation $H_k\approx[\nabla^2 f]^{-1}$, initialisée à $I$ :

$$d_k=-H_k\nabla f(x_k),\qquad x_{k+1}=x_k+\alpha_k d_k$$
$$s_k=x_{k+1}-x_k,\quad y_k=\nabla f(x_{k+1})-\nabla f(x_k),\quad \rho_k=\frac1{y_k^Ts_k}$$
$$H_{k+1}=(I-\rho_k s_ky_k^T)\,H_k\,(I-\rho_k y_ks_k^T)+\rho_k s_ks_k^T$$

La mise à jour n'est faite que si $y_k^Ts_k>0$ (condition de courbure) : $H_k$ reste alors symétrique définie positive.

In [11]:
def quasi_newton_bfgs(f, grad, x0, tol=1e-6, max_iter=1_000,
                      verbose=False, n_affiche=10):
    """
    On maintient H_k ~ (Hessienne)^-1, initialisée à l'identité.

        d_k     = -H_k grad f(x_k)
        x_{k+1} = x_k + alpha_k d_k            (alpha_k : Armijo)
        s_k     = x_{k+1} - x_k
        y_k     = grad f(x_{k+1}) - grad f(x_k)
        rho_k   = 1 / <y_k, s_k>
        H_{k+1} = (I - rho s y^T) H_k (I - rho y s^T) + rho s s^T

    La mise à jour n'est faite que si <y_k, s_k> > 0 (condition de courbure),
    ce qui garantit que H_{k+1} reste symétrique définie positive.
    """
    x = np.array(x0, dtype=float)
    n = len(x)
    H = np.eye(n)
    g = grad(x)
    hist = [x.copy()]
    if verbose:
        _affiche_entete("Quasi-Newton BFGS")

    for k in range(max_iter):
        ng = np.linalg.norm(g)
        if ng < tol:
            if verbose:
                _affiche_ligne(k, x, f(x), ng, None)
            return Resultat(x, f(x), ng, k, True, hist, H)
        d = -H @ g
        if g @ d >= 0:                        # sécurité : on repart de l'identité
            H = np.eye(n)
            d = -g
        alpha = backtracking_armijo(f, g, x, d)
        if verbose and k < n_affiche:
            _affiche_ligne(k, x, f(x), ng, alpha)

        x_new = x + alpha * d
        g_new = grad(x_new)
        s = x_new - x
        y = g_new - g
        sy = y @ s
        if sy > 1e-12:
            rho = 1.0 / sy
            I = np.eye(n)
            H = (I - rho * np.outer(s, y)) @ H @ (I - rho * np.outer(y, s)) \
                + rho * np.outer(s, s)
        x, g = x_new, g_new
        hist.append(x.copy())

    return Resultat(x, f(x), np.linalg.norm(g), max_iter, False, hist, H)

### Exemple 4 — BFGS sur la fonction de Rosenbrock

$$f(x,y)=(1-x)^2+100(y-x^2)^2,\qquad X_0=(-1.2,\,1),\qquad X^*=(1,1)$$

Fonction réputée difficile : vallée étroite et courbe.

In [12]:
expr_rosen = (1 - x) ** 2 + 100 * (y - x**2) ** 2
f_ros, grad_ros = fonction_depuis_sympy(expr_rosen, [x, y])

In [13]:
titre("EXEMPLE 4 - QUASI-NEWTON (BFGS)")
print("f(x,y) = (1-x)^2 + 100 (y - x^2)^2,   X0 = (-1.2, 1),   H0 = I")

print("\n--- Deroulement a la main de l'iteration 0 ---")
X0 = np.array([-1.2, 1.0])
g0 = grad_ros(X0)
print(f"grad f(X0) = {g0}")
print(f"d0 = -H0 grad f(X0) = {-g0}   (H0 = I : premiere iteration = descente de gradient)")
a0 = backtracking_armijo(f_ros, g0, X0, -g0)
X1 = X0 - a0 * g0
g1 = grad_ros(X1)
s, yv = X1 - X0, g1 - g0
print(f"alpha0 (Armijo) = {a0:.6f}")
print(f"X1 = {X1}")
print(f"s0 = X1 - X0 = {s}")
print(f"y0 = grad f(X1) - grad f(X0) = {yv}")
print(f"<y0, s0> = {yv @ s:.6f} > 0  -> mise a jour BFGS autorisee")
rho = 1 / (yv @ s)
I = np.eye(2)
H1 = (I - rho * np.outer(s, yv)) @ (I - rho * np.outer(yv, s)) + rho * np.outer(s, s)
print(f"H1 = (I - rho s y^T)(I - rho y s^T) + rho s s^T =\n{H1}")

res = quasi_newton_bfgs(f_ros, grad_ros, X0, verbose=True, n_affiche=12)
print(f"\nResultat : X = {res.x}  en {res.n_iter} iterations")
print(f"Solution exacte : [1. 1.]")
hess = np.array(sp.hessian(expr_rosen, (x, y)).subs({x: 1, y: 1}), dtype=float)
print(f"\nH final (approx. de l'inverse Hessienne) :\n{res.H_inverse}")
print(f"Inverse exacte de la Hessienne en (1,1) :\n{np.linalg.inv(hess)}")


EXEMPLE 4 - QUASI-NEWTON (BFGS)
f(x,y) = (1-x)^2 + 100 (y - x^2)^2,   X0 = (-1.2, 1),   H0 = I

--- Deroulement a la main de l'iteration 0 ---
grad f(X0) = [-215.6  -88. ]
d0 = -H0 grad f(X0) = [215.6  88. ]   (H0 = I : premiere iteration = descente de gradient)
alpha0 (Armijo) = 0.000977
X1 = [-0.989453  1.085938]
s0 = X1 - X0 = [0.210547 0.085938]
y0 = grad f(X1) - grad f(X0) = [253.93803  109.384003]
<y0, s0> = 62.866046 > 0  -> mise a jour BFGS autorisee
H1 = (I - rho s y^T)(I - rho y s^T) + rho s s^T =
[[ 0.15727  -0.363182]
 [-0.363182  0.843922]]

Quasi-Newton BFGS
   k | x_k                          |         f(x_k) |    ||grad|| |    alpha_k
--------------------------------------------------------------------------------
   0 | [-1.2  1. ]                  |    24.20000000 |   2.329e+02 |    0.00098
   1 | [-0.98945  1.08594]          |     5.10111266 |   4.390e+01 |    0.12500
   2 | [-0.77235  0.57059]          |     3.20842357 |   1.266e+01 |    1.00000
   3 | [-0.74129  0

## 5. Comparaison finale sur Rosenbrock

Même point de départ, même tolérance : le nombre d'itérations illustre l'intérêt des méthodes de direction conjuguée et quasi-Newton.

In [14]:
titre("COMPARAISON sur Rosenbrock, X0 = (-1.2, 1), tol = 1e-6")
X0 = np.array([-1.2, 1.0])
algos = {
    "Gradient pas fixe (alpha=1e-3)":
        lambda: descente_gradient_pas_fixe(f_ros, grad_ros, X0, alpha=1e-3,
                                           max_iter=200_000),
    "Gradient pas variable (Armijo)":
        lambda: descente_gradient_pas_variable(f_ros, grad_ros, X0,
                                               max_iter=200_000),
    "Gradient conjugue (PR+)":
        lambda: gradient_conjugue_non_lineaire(f_ros, grad_ros, X0,
                                               max_iter=200_000),
    "Quasi-Newton (BFGS)":
        lambda: quasi_newton_bfgs(f_ros, grad_ros, X0, max_iter=200_000),
}
print(f"{'Algorithme':<34} | {'iterations':>10} | {'converge':>8} | solution")
print("-" * 80)
for nom, run in algos.items():
    r = run()
    print(f"{nom:<34} | {r.n_iter:>10} | {str(r.converge):>8} | {r.x}")


COMPARAISON sur Rosenbrock, X0 = (-1.2, 1), tol = 1e-6
Algorithme                         | iterations | converge | solution
--------------------------------------------------------------------------------
Gradient pas fixe (alpha=1e-3)     |      32076 |     True | [0.999999 0.999998]
Gradient pas variable (Armijo)     |      13756 |     True | [0.999999 0.999998]
Gradient conjugue (PR+)            |       1033 |     True | [1. 1.]
Quasi-Newton (BFGS)                |         34 |     True | [1. 1.]


## Conclusion

| Méthode | Coût par itération | Vitesse |
|---|---|---|
| Gradient pas fixe | très faible | lente, sensible au choix de α |
| Gradient pas variable | faible (+ backtracking) | lente sur vallées étroites |
| Gradient conjugué | faible | bien plus rapide, finie ($n$ itérations) en quadratique |
| BFGS | $O(n^2)$ | superlinéaire, la plus rapide ici |